# TCC Nível Guaíba - Exploração de Dados

## Bibliotecas

In [ ]:
# ML Libraries;
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
from pyod.models.pca import PCA
from pyod.models.ecod import ECOD

# Ploting Libraries;
import os
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ydata_profiling import ProfileReport

In [ ]:
# Internal imports;
from src.db_handler import DBConnection
from src.util import convert_to_float, STATION_COLS, AGG_DICT, START_DATE, END_DATE, STATION_CODES
from source_frontend.make_station_comparison import make_station_comparison
from source_frontend.make_outlier_view import compare_outlier_detectors, compare_multiple_contamination_levels
from src.etl.data_transformation import fill_gaps, aggregate_data, melt_dataframe
from src.etl.data_cleaning import clean_dataframe
from src.etl.outlier_detection import outlier_removal
from src.etl.data_imputation import feature_imputation

# # Google Colab Imports;
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# sys.path.append('/content/drive/MyDrive/TCC Juliano Machado')

# from src.db_handler import DBConnection
# from src.util import convert_to_float, STATION_COLS, AGG_DICT, START_DATE, END_DATE, STATION_CODES
# from make_station_comparison import make_station_comparison
# from make_outlier_view import compare_outlier_detectors, compare_multiple_contamination_levels
# from data_transformation import fill_gaps, aggregate_data, melt_dataframe
# from data_cleaning import clean_dataframe
# from outlier_detection import outlier_removal
# from data_imputation import feature_imputation

## Introdução

* Este notebook explora os dados das estações hidrometeorológicas obtidos via API do HidroWeb, serviço do Sistema Nacional de Informações sobre Recursos Hídricos (SNIRH), mantido pela Agência Nacional de Águas (ANA). A base contém informações de mais de 23 mil estações de monitoramento distribuídas pelo Brasil, com registros de variáveis hidrometeorológicas, como nível, vazão, chuva, temperatura, entre outros; 

* O objetivo é comparar modelos de predição do nível da água no Rio Guaíba. Para isso, são utilizadas as Séries Históricas desse nível, complementadas por dados meteorológicos como chuva, chuva acumulada e temperatura, também obtidos do HidroWeb;  

* As estações analisadas pertencem à Bacia Hidrográfica do Guaíba, no Rio Grande do Sul, abrangendo os principais rios afluentes — Taquari, Caí, Gravataí, Sinos e Jacuí — a fim de garantir ampla cobertura espacial e boa qualidade dos dados;  

* Documentação da API (Swagger): [https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html#/](https://www.ana.gov.br/hidrowebservice/swagger-ui/index.html#/)

## API HidroWeb

### Endpoints utilizados
* As possíveis estações foram obtidas do endpoint: https://www.ana.gov.br/hidrowebservice/EstacoesTelemetricas/HidroInventarioEstacoes/v1

* As séries detalhadas foram obtidas do endpoint: https://www.ana.gov.br/hidrowebservice/EstacoesTelemetricas/HidroinfoanaSerieTelemetricaDetalhada/v1

In [ ]:
nodes = ["Estações na bacia", "Operando", "Telemétrica", "Histórico suficiente", "Utilizáveis", "Selecionadas", "Filtradas"]
source = [0, 0, 1, 1, 2, 2, 3, 3, 4]
target = [1, 6, 2, 6, 3, 6, 4, 6, 5]
value  = [128, 130, 43, 85, 33, 10, 15, 18, 9, 6]

fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    node=dict(
        pad=40,
        thickness=20,
        line=dict(color="black", width=0.3),
        label=nodes,
        color=["lightgreen"]*6 + ["rgba(200,50,50,0.6)"]  # red for filtered-out
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=[
            "rgba(100,200,100,0.5)" if t != 6 else "rgba(200,100,100,0.4)"
            for t in target
        ]
    )
)])

fig.update_layout(
    title_text="Estações Selecionadas",
    font=dict(size=14),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font_color="black"
)
fig.show()

* Foram selecionadas as estações pertencentes à Bacia Hidrográfica do Guaíba, com:  
  * Bacia = 'ATLÂNTICO, TRECHO SUDESTE'  
  * Sub_Bacia = 'LAGOA DOS PATOS'  
  * Bacia_Codigo = '8'  
  * Sub_Bacia_Codigo = '87'  

* Rios afluentes considerados (`Rio_Codigo`):  
  * Guaíba = '87200000'  
  * Taquari = '86001000'  
  * Caí = '87220000'  
  * Gravataí = '87240000'  
  * Sinos = '87230000'  
  * Jacuí = '85001000'  

* O total inicial era de 258 estações. Após filtrar por **Operando = '1'** e **Tipo_Estacao_Telemetrica = '1'**, restaram **43 estações**.  
  Excluíram-se ainda estações com finalidades específicas (energéticas, piezométricas etc.);
  ```
  Rio_Codigo IN ('87200000','86001000','87220000','87240000','87230000','85001000')
  AND Operando IN ('1')
  AND Tipo_Estacao_Telemetrica IN ('1');
  ```

* As séries históricas dessas estações foram analisadas quanto à disponibilidade, qualidade, lacunas, duração e representatividade;

* A estação-alvo das predições é CAIS MAUÁ C6 (87450004), no Rio Guaíba, usada como referência durante as enchentes de 2024. Após o dano aos sensores em 02/05/2024, seus dados passaram a ser continuados pela estação USINA DO GASÔMETRO (CHEIA 2024);

* Estações selecionadas:
  * Guaíba: CAIS MAUÁ C6 ('87450004'), USINA DO GASÔMETRO (CHEIA 2024) ('87444000')
  * Taquari: MUÇUM ('86510000'), ENCANTADO ('86720000')
  * Caí: LINHA GONZAGA ('87150000'), BARCA DO CAÍ ('87170000')
  * Gravataí: PASSO DAS CANOAS - AUXILIAR ('87399000')
  * Sinos: SÃO LEOPOLDO ('87382000'), CAMPO BOM ('87380000')
  * Jacuí: RIO PARDO ('85900000')

* Para uniformizar as séries, adotou-se `2018-08-01` como data inicial comum, totalizando cerca de seis anos de dados telemétricos, com registros a cada 15 minutos.
Devido ao limite de 30 dias imposto pela API, o processo de coleta foi realizado em chunks mensais posteriormente concatenados;

## Visualização Dados

In [ ]:
# Initialize the Connection;
db = DBConnection()

# Query data for all required stations;
query = '''
    SELECT * FROM HidroinfoanaSerieTelemetricaDetalhada_v1
    WHERE codigoestacao IN (
        '87450004', '87444000', '87399000', '87382000', 
        '87380000', '86510000', '86720000', '87150000', 
        '87170000', '85900000'
    )
'''
df = db.run(query=query)['result']

# Ensure codigoestacao is in lowercase
if 'CodigoEstacao' in df.columns:
    df = df.rename(columns={'CodigoEstacao': 'codigoestacao'})

# Merge guaiba_1 and guaiba_2 into a single dataframe;
guaiba_merged = df[df['codigoestacao'].isin(['87450004', '87444000'])].copy()
guaiba_merged['codigoestacao'] = '87450004'

# Remove original guaiba stations from df and concatenate the merged one;
df = df[~df['codigoestacao'].isin(['87450004', '87444000'])]
df = pd.concat([df, guaiba_merged], ignore_index=True)


In [ ]:
# Query data;
query = {
    'data_stations_cleaned':        'SELECT * FROM data_stations_cleaned',
    'data_stations_filled':         'SELECT * FROM data_stations_filled',
    'data_stations_missing':        'SELECT * FROM data_stations_missing',
    'data_stations_aggregated':     'SELECT * FROM data_stations_aggregated',
    'data_stations_outlier':        'SELECT * FROM data_stations_outlier',
    'data_stations_imputed':        'SELECT * FROM data_stations_imputed',
    'data_stations_melted':         'SELECT * FROM data_stations',
    }
dataframe = db.run(query=query)

In [ ]:
# Convert values and cut the dataframe to a time range where most data is available;
# df_cleaned = clean_dataframe(df=df)
df_cleaned = dataframe['data_stations_cleaned']
df_cleaned

In [ ]:
# # Automated data profiling;
# profile = ProfileReport(df_cleaned, title="Profiling Report")
# profile.to_notebook_iframe()

### Valores faltantes

* Pode-se ver que certos atributos estão consistentemente faltando do dataset. Em especial, todos aqueles relacionados à Cota que não o 'Cota_Adotada'. Estes atributos podem ser considerados secundários na análise e serão tratadados de tal forma;
    * Na análise à seguir, veremos que todos os 'Cota_*' são altamente relacionados com os valores de 'Cota_Adotada' e, como o mesmo é o nosso alvo de análises, usaremos valores de 'Cota_Manual' e 'Cota_Sensor' para preencher valores faltantes de 'Cota_Adotada';

* Além disso, 'Pressao_Atmosferica' e 'Temperatura_Agua' infelizmente apresentam muitos valores faltantes para serem de qualquer uso nessa análise e logo serão desconsiderados daqui pra frente;

* Por fim, vale notar que estes valores são a média para todas as estações e, quando tratando cada uma de forma individual, a quantidade de valores faltantes varia bastante. Buscou-se coletar dados de estações que estivessem com o mínimo de valores faltantes possível;

In [ ]:
# Percentage of Null values
missing = 100 * df_cleaned.isnull().sum() / len(df_cleaned)
missing = missing[missing > 0].sort_values(ascending=True)

plt.figure(figsize=(12, 6))
ax = missing.plot(kind='barh')

plt.xlabel('Porcentagem de valores faltantes')
plt.title('Valores faltantes')

# Add value labels at the end of each bar;
for i, value in enumerate(missing):
    ax.text(value + 0.5, i, f'{value:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### Correlação

* A análise de correlação mostra que as colunas **'Cota_Manual'**, **'Cota_Sensor'** e **'Cota_Display'** têm alta correlação com o target **'Cota_Adotada'**;
    * A variável **'Cota_Manual'** apresenta mais de **99% de valores nulos**, e seus poucos valores não nulos ocorrem majoritariamente quando **'Cota_Adotada'** está ausente. Isso indica que essas medições manuais servem para substituir as automáticas durante períodos de manutenção ou falha do sensor;
    * As variáveis **'Cota_Sensor'** e **'Cota_Display'** são, na maioria dos casos, idênticas à **'Cota_Adotada'**, o que é consistente com o fato de que as leituras são predominantemente obtidas de sensores automáticos;

* A variável **Vazao_Adotada** apresenta alta correlação com **Cota_Adotada**, o que é consistente com o comportamento hidrodinâmico: o aumento da vazão ($m^3/s$) em um leito fixo eleva o nível da água até atingir as áreas de várzea adjacentes. Essa variável será fundamental para a predição, dada sua forte relação com o target e baixa correlação com outras variáveis — indicando que traz informação complementar relevante ao modelo;

* No entanto, há uma alta taxa de valores ausentes (>15%), cuja magnitude varia entre estações. A estação alvo (**Guaíba**) não possui registros de vazão, o que limita seu uso direto nessa localidade;

In [ ]:
%matplotlib inline
a = df_cleaned.reset_index()
columns = [col for col in df_cleaned.columns if not col.endswith('_Status')]
columns = list(set(columns) - set(['Pressao_Atmosferica', 'Temperatura_Agua']))
a = a[columns]

plt.figure(figsize=(25, 15))
sns.heatmap(
    data=a.corr(),
    annot=True,
    fmt=".1f",
    annot_kws={"size": 10},
)
plt.xticks(rotation=45, ha='right')  # tilt x-axis labels for readability
plt.yticks(rotation=0)
plt.tight_layout()
plt.show("png")

# Variáveis selecionadas

* Baseado na análise de correlações acima e na quantidade de dados faltantes, foram selecionadas as seguintes variáveis para os modelos de ML:
- **'Cota_Adotada'**: Correspontente ao nível do rio em centímetros. Os dados faltantes serão completados utilizando 'Cota_Manual' e 'Cota_Sensor' quando estes estão disponíveis;
- **'Chuva_Adotada'**:  Correspondente à precipitação detectada na estação desde o último registro em milímetros;
- **'Chuva_Acumulada'**: Correspondente à quantidade de precipitação acumulada em um período de 1 mês em milímetros. A consistência desta medida será verificada e, caso necessário, será recriada a partir de 'Chuva_Adotada', possivelmente com outros intervalos de acumulação;
- **'Temperatura_Interna'**: Correspondente à temperatura atmosférica medida no equipamento em °C;
- **'Vazao_Adotada'**: Correspondente ao fluxo de água registrado pelo sensor em $\frac{m^3}{s}$. Esta variável não está disponível para a estação alvo;

### Comparação entre as estações

* Ao comparar os dados de nível da água entre as estações, nota-se um dos principais problemas dos dados hidrometeorológicos do HidroWeb: **gaps de dados**. Todas as estações apresentam falhas, em diferentes intensidades, com períodos sem registros;

* Na estação **RIO PARDO ('85900000')** evidencia-se bem esse problema. Seus gaps são extensos, com o maior de **nov/2020** a **jul/2021**. Nesse intervalo, há registros pontuais, mas sem a frequência regular de **15 minutos** observada no restante do dataset;
    * Nem todos os gaps apresentam registros esparsos. Em alguns períodos, os dados simplesmente não existem;

In [ ]:
make_station_comparison(df=df_cleaned)

<!-- ![AA](/home/juju/Documents/Nivel_TCC/data/comparação_nível_estações.png) -->

## Preenchimento de falhas

* Para tratar as falhas de frequência presentes nas estações, adotaram-se duas abordagens:
    * **Interpolação linear** para gaps pequenos (≤ 2h, ou até 8 intervalos de 15 min). Esse limite é arbitrário e pode ser reajustado;
    * **Imputação iterativa** para gaps grandes, usando `IterativeImputer` com `RandomForestRegressor`, robusto a relações não lineares;

* Para a interpolação linear, foi necessário criar uma timeline continua de 15 minutos para todas as estações, para que estes valores que não estão presente, mas que não são fáceis de vizualizar, pudessem ser preenchidos;

* Para a variável 'Cota_Adotada' também foi realizada uma imputação manual dos valores faltantes usando os valores de **'Cota_Manual'** e **'Cota_Sensor'**;

* Com a linha temporal completa para cada estação, o número de valores faltantes aumenta consideravelmente. Abaixo a quantidade de valores faltantes de **'Cota_Adotada'** por estação:

In [ ]:
# Fill the data gaps;
# df_filled, missing_values = fill_gaps(df=df_cleaned, max_fill_steps=8)
df_filled = dataframe['data_stations_filled']
missing_values = dataframe['data_stations_missing']
df_filled

In [ ]:
# Create grouped bar chart with two bars per station;
plot = sns.catplot(
    data=missing_values,
    kind='bar',
    x='station_id',
    y='missing_percentage',
    hue='period',
    height=6,
    aspect=2
)

# Add labels to both groups of bars;
for ax in plot.axes.flat:
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', fontsize=10);

plot.set_xlabels('Estação')
plot.set_ylabels('Dados faltantes de Cota_Adotada (%)')
plot.fig.suptitle('Dados faltantes - Antes e depois do gap filling')
plot.set_xticklabels(rotation=45, ha='right')
plt.tight_layout()
plt.show()

* O processo de imputação com `IterativeImputer` será realizado posteriormente, após o tratamento e substituição dos outliers, que seguirão a mesma abordagem.


### Colunas de Status

* Cada coluna de informação é acompanhada por uma coluna de Status, referente ao estado da coleta destes valores pelos sensores da estação;

* Foi seguida a convenção adotada pelo SNIRH, com a inclusão de uma flag para valores que foram imputados e/ou valores faltantes;

    * Status Codes Quality Control Convention: 0=Normal, 1=Suspeito, 2=Ruim, 3=Muito Ruim, 4=Preenchido/Faltante, 5=Outlier;

## Agregação de dados

* A frequência original de 15 minutos à cada registro dos dados do HidroWeb não é ideal para realizar predições tanto de longo prazo, como de curto prazo. Em ambos os casos, seria necessário um horizonte de predição muito grande para realizar qualquer predição relevante temporalmente.

* Logo, foi necessário criar um método para agrupar os dados com uma frequência qualquer;

In [ ]:
# Aggregate the data to the desired frequency;
# df_agg = aggregate_data(df=df_filled, frequency='h')
df_agg = dataframe['data_stations_aggregated']
df_agg

In [ ]:
make_station_comparison(df=df_agg)

## Visualização Outliers
* Um dos pontos abordados no processo de limpeza é a inserção manual de valores faltantes de **Cota_Adotada** a partir de **Cota_Manual** e **Cota_Sensor**. Esse procedimento é essencial para reduzir o volume de dados ausentes em algumas estações. Na estação **Jacuí (RIO PARDO)** (**85900000**), por exemplo, cerca de 40% dos valores estavam faltando antes desse tratamento, em grande parte devido a gaps na frequência original de 15 minutos. Após o processamento, o percentual de faltantes cai para aproximadamente 20%, tornando o conjunto muito mais manejável;  

* O principal efeito colateral desse processo é a inserção de **outliers**, perceptíveis nas estações analisadas. Esses valores geralmente decorrem de erros de escala do sensor. Ainda assim, o ganho em cobertura de dados, especialmente em estações periféricas, compensa a introdução de outliers, que podem ser facilmente identificados e imputados posteriormente;

In [ ]:
# # Analyze all stations combined;
# fig = compare_outlier_detectors(
#     df=df_agg,
#     threshold_method='iqr',
#     show_plot=True,
#     save_plot=False
# )

# Tratamento de Outliers
* Para identificar os Outliers foi utilizada a biblioteca `PyOD`, especializada em deteção de Outliers em dados com multiplas variáveis com os algoritmos `ECOD` e `PCA`, sendo o primeiro um algoritmo focado na análise de eventos extremos nas caudas de distribuições cumulativas e o segundo um algoritmo que busca outliers em dados reconstruídos a partir de projeções de menor dimensionalidade dos mesmos;

* Estes dois algoritmos criam um score de outlier para cada valor. A combinação de ambos os scores é utilizada para determinar se um valor é um outlier ou não usando IQR (Range interquartil). Este método é utilizado em combinação com o método "Tukey Fence", que trata qualquer ponto acima de Q3 + 1.5*(Q3-Q1) ou abaixo de Q3 + 1.5*(Q3-Q1) como um outlier. Este método foi escolhido por conseguir escolher dinamicamente os outliers de cada estação, ou seja, sem utilizar um valor fixo, dada a diferença na quantidade de outliers entre cada estação;

* Os valores determinados como outliers foram removidos, deixando somente valores nulos no seu lugar. Isto foi feito para que os valores nulos já existentes e os outliers pudessem ser tratados com a mesma lógica, salvando poder computacional;

In [ ]:
# Identify and remove Outliers;
# df_out = outlier_removal(df=df_agg)
df_out = dataframe['data_stations_outlier']
df_out

In [ ]:
make_station_comparison(df=df_out)

# Imputação de Dados
* Para a imputação de dados, foi escolhido um método de imputação iterativo, no qual os valores faltantes são estimados através dos valores presentes correspondentes com um estimador customizado;
    * Foi escolhido um `RandomForestRegressor` devido a sua robustes e capacidade de criar árvores mais profundas do que outros métodos; 

In [ ]:
# Feature Imputation - IteractiveImputer;
# df_imp = feature_imputation(df=df_out)
df_imp = dataframe['data_stations_imputed']
df_imp

In [ ]:
make_station_comparison(df=df_imp)

# Transformar formato comprido
* Até esse ponto o dataframe está no formato longo, ou seja, cada linha indica um registro para uma estação com os seus atributos. Dessa forma, temos várias linhas para o mesmo instante de tempo;

* No formato comprido, estaremos transformando o dataframe em um único registro de tempo para todas as estações. Em cada um destes registros, teremos todas as informações de todas as estações;

In [ ]:
df_melt = dataframe['data_stations']
df_melt

# Próximos passos

* Testar spline ao invés de feature imputing;

* Feature Scalling em uma Pipeline;

* Transformação de dados para formato de GPU com CuPy;

* Introdução de timelags como features;

* Análise de Feature Importance;

* Versão inicial de modelos;